In [55]:
import pandas as pd
from nrclex import NRCLex
from sklearn.metrics import classification_report, accuracy_score


In [56]:
df_songs = pd.read_csv('dataset.csv')

In [57]:
def get_nrc_emotions(text):
    emotions = NRCLex()
    emotions.load_raw_text(text)
    return emotions.affect_frequencies


In [58]:
emotions = df_songs['lyrics_cleaned'].apply(lambda x: pd.Series(get_nrc_emotions(x)))

df_songs = pd.concat([df_songs,emotions],axis=1)

In [59]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Define your features (the 10 NRC categories)
features = ['fear', 'anger', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust', 'positive', 'negative']
X = df_songs[features]
y = df_songs['parent_genre']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# 2. Get the overall accuracy score
accuracy = accuracy_score(y_test, predictions)
print(f"Overall Model Accuracy: {accuracy:.2%}")
print("-" * 30)

# 3. Print the full Summary (Precision, Recall, F1)
print("Classification Report:")
print(classification_report(y_test, predictions))

Overall Model Accuracy: 15.74%
------------------------------
Classification Report:
                  precision    recall  f1-score   support

       Asian Pop       0.05      0.06      0.05        96
       Classical       0.01      0.12      0.02        33
      Electronic       0.35      0.03      0.05       880
    Folk/Country       0.18      0.09      0.12       333
   Hip-Hop & R&B       0.03      0.38      0.05        29
    Jazz & Blues       0.04      0.15      0.07        52
           Metal       0.29      0.59      0.39       435
             Pop       0.13      0.01      0.03       289
Reggae/Caribbean       0.02      0.04      0.03        76
            Rock       0.19      0.05      0.08       525
       Soul/Funk       0.11      0.12      0.12       147
  World/Regional       0.22      0.53      0.31       180

        accuracy                           0.16      3075
       macro avg       0.14      0.18      0.11      3075
    weighted avg       0.23      0.16      

In [60]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Define your features (the 10 NRC categories)
features = ['fear', 'anger', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust', 'positive', 'negative']
X = df_songs[features]
y = df_songs['parent_genre']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

def train_random_forests(df: pd.DataFrame, ests: int = 500, max_deep:int =100):
        # Define your features (the 10 NRC categories)
    features = ['fear', 'anger', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust', 'positive', 'negative']
    X = df[features]
    y = df['parent_genre']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    model = RandomForestClassifier(
            n_estimators=100,        # Number of trees (100 is a good starting point)
            class_weight='balanced', # Still crucial for your imbalanced genres
            n_jobs=-1,               # PRO TIP: Uses all your CPU cores to speed up training
            random_state=42,
            max_depth=max_deep
        )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    # 2. Get the overall accuracy score
    accuracy = accuracy_score(y_test, predictions)
    print(f"Overall Model Accuracy: {accuracy:.2%}")
    print("-" * 30)

    # 3. Print the full Summary (Precision, Recall, F1)
    print("Classification Report:")
    print(classification_report(y_test, predictions))
    return model

for x in range(30,110,10):
    train_random_forests(df_songs,x)

Overall Model Accuracy: 34.70%
------------------------------
Classification Report:
                  precision    recall  f1-score   support

       Asian Pop       0.75      0.03      0.05       118
       Classical       0.67      0.08      0.14        51
      Electronic       0.33      0.73      0.46       841
    Folk/Country       0.30      0.15      0.20       312
   Hip-Hop & R&B       0.67      0.08      0.14        25
    Jazz & Blues       0.44      0.06      0.11        65
           Metal       0.40      0.45      0.42       482
             Pop       0.41      0.11      0.18       279
Reggae/Caribbean       0.50      0.02      0.05        84
            Rock       0.24      0.14      0.18       466
       Soul/Funk       0.50      0.07      0.13       165
  World/Regional       0.50      0.36      0.42       187

        accuracy                           0.35      3075
       macro avg       0.48      0.19      0.21      3075
    weighted avg       0.38      0.35      

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

genres = df_songs['parent_genre'].unique()

def fit_rand_forest_model(X_train,y_train):
    # Create a pipeline: Vectorizer -> Logistic Regression
    # We use 'lbfgs' solver which is good for multiclass problems
    model_pipeline = RandomForestClassifier(
            n_estimators=100,        # Number of trees (100 is a good starting point)
            class_weight='balanced', # Still crucial for your imbalanced genres
            n_jobs=-1,               # PRO TIP: Uses all your CPU cores to speed up training
            random_state=42,
            max_depth=30
        )

    # Train the model
    print("Training model...")
    model_pipeline.fit(X_train, y_train)
    return model_pipeline

def fit_log_reg_model(X_train,y_train):
    # Create a pipeline: Vectorizer -> Logistic Regression
    # We use 'lbfgs' solver which is good for multiclass problems
    model_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=9000, stop_words='english')),
        ('clf', LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced'))
    ])

    # Train the model
    print("Training model...")
    model_pipeline.fit(X_train, y_train)
    return model_pipeline

def output_model(df: pd.DataFrame):
    # Split the data into features (X) and target (y)
    features = ['fear', 'anger', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust', 'positive', 'negative']
    X = df[features+['lyrics_cleaned']]
    y = df['parent_genre']
    y_unique = y.unique()

    
    # Split into training and test sets (80% training, 20% testing)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    large_model = fit_log_reg_model(X_train['lyrics_cleaned'],y_train)
    # Make predictions
    predictions = large_model.predict(X_test['lyrics_cleaned'])

    # Get the probabilities
    probs = large_model.predict_proba(X_test['lyrics_cleaned'])

    # Get the genre names from the model
    genres = large_model.classes_

    # Create a DataFrame where each column is a Genre
    y_prob = pd.DataFrame(probs, columns=genres, index=X_test.index)
    
    # This creates a dictionary of {Genre: Probability} for the top 3
    y_prob['predicted'] = y_prob.apply(
        lambda row: row.nlargest(3).to_dict(), 
        axis=1
    )

    df_pred = pd.DataFrame({
        'actual': y_test,
        'lyrics_cleaned':X_test['lyrics_cleaned']
    })

    df_pred = df_pred.join(y_prob['predicted'])

    combined_xy_train = X_train[features].join(y_train)
    combined_xy_test = X_test[features].join(y_test)
    combos_models = {}
    counter = 1
    for i in range(len(y_unique)):
        for j in range(i+1,len(y_unique)):
            for k in range(j+1,len(y_unique)):
                combo_tuple = (y_unique[i],y_unique[j],y_unique[k])
                print("Currently on combo:",combo_tuple, f"{counter}/286")
                sub_x_y_train = combined_xy_train[combined_xy_train['parent_genre'].isin([y_unique[i],y_unique[j],y_unique[k]])]
                sub_x_y_test = combined_xy_test[combined_xy_test['parent_genre'].isin([y_unique[i],y_unique[j],y_unique[k]])]
                sub_model = fit_rand_forest_model(sub_x_y_train[features],sub_x_y_train['parent_genre'])
                sub_predictions = sub_model.predict(sub_x_y_test[features])
                combos_models[combo_tuple] = (sub_model,accuracy_score(sub_x_y_test['parent_genre'],sub_predictions))
                counter += 1
    
    return combos_models,df_pred

model_dict,predicts = output_model(df_songs)

Training model...
Currently on combo: ('World/Regional', 'Rock', 'Electronic') 1/286
Training model...
Currently on combo: ('World/Regional', 'Rock', 'Asian Pop') 2/286
Training model...
Currently on combo: ('World/Regional', 'Rock', 'Metal') 3/286
Training model...
Currently on combo: ('World/Regional', 'Rock', 'Folk/Country') 4/286
Training model...
Currently on combo: ('World/Regional', 'Rock', 'Jazz & Blues') 5/286
Training model...
Currently on combo: ('World/Regional', 'Rock', 'Latin') 6/286
Training model...
Currently on combo: ('World/Regional', 'Rock', 'Classical') 7/286
Training model...
Currently on combo: ('World/Regional', 'Rock', 'Reggae/Caribbean') 8/286
Training model...
Currently on combo: ('World/Regional', 'Rock', 'Soul/Funk') 9/286
Training model...
Currently on combo: ('World/Regional', 'Rock', 'Hip-Hop & R&B') 10/286
Training model...
Currently on combo: ('World/Regional', 'Rock', 'Pop') 11/286
Training model...
Currently on combo: ('World/Regional', 'Electronic',

In [12]:
predicts[features] = df_songs[features]

In [40]:
import warnings

# Only silences the parallel/delayed warning from sklearn
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.utils.parallel")

predictions = []
i = 1
for dict_top_3,index in zip(predicts['predicted'].to_list(),predicts.index):
    for model_key,model in model_dict.items():
        correct_model = True
        for top_3_key in dict_top_3.keys():
            if top_3_key not in model_key:
                correct_model=False
        if correct_model:
            single_row = predicts.loc[[index],features]
            probs = model[0].predict_proba(single_row[features])[0]
            genres = model[0].classes_
            current_max_prob = 0
            current_max_genre = ''
            for i in range(len(probs)):
                combined_prob = probs[i]*dict_top_3[genres[i]]
                if combined_prob > current_max_prob:
                    current_max_prob = combined_prob
                    current_max_genre = genres[0]
            predictions.append(current_max_genre)
            print(f"\rPredicting row: ({i+1}/{len(predicts)})", end='', flush=True)
            i +=1

predicts['final'] = predictions

Predicting row: (3/3087)

In [41]:
predicts['correct'] = predicts['actual']==predicts['final']

In [ ]:
predictions = []
for dict_top_3,lyrics in zip(predicts['predicted'].to_list(),predicts['lyrics_cleaned'].to_list()):
    for model_key,model in model_dict.items():
        correct_model = True
        for top_3_key in dict_top_3.keys():
            if top_3_key not in model_key:
                correct_model=False
        if correct_model:
            single_row = predicts.loc[[index],features]
            probs = model[0].predict_proba(single_row[features])[0]
            genres = model[0].classes_
            current_max_prob = 0
            current_max_genre = ''
            for i in range(len(probs)):
                combined_prob = probs[i]*dict_top_3[genres[i]]
                if combined_prob > current_max_prob:
                    current_max_prob = combined_prob
                    current_max_genre = genres[0]
            predictions.append(current_max_genre)

In [42]:
avg_acc = 0.0
for key, accuracy in model_dict.items():
    avg_acc += accuracy[1]

avg_acc = avg_acc/len(model_dict)
print(avg_acc)

0.7049645426119218


In [54]:
df_not = pd.DataFrame()
for feat in features:
    df_not[feat] = df_songs[[feat]+['parent_genre']].groupby('parent_genre').mean()

print(df_not.map('{:.4f}'.format))

                    fear   anger anticipation   trust surprise sadness  \
parent_genre                                                             
Asian Pop         0.0956  0.0777       0.1006  0.0939   0.0497  0.0878   
Classical         0.0725  0.0494       0.1199  0.1196   0.0511  0.0723   
Electronic        0.0985  0.0701       0.1067  0.0863   0.0461  0.0984   
Folk/Country      0.0790  0.0600       0.1112  0.0974   0.0490  0.0993   
Hip-Hop & R&B     0.1013  0.0961       0.0944  0.0858   0.0425  0.0952   
Jazz & Blues      0.0849  0.0564       0.1186  0.0943   0.0472  0.0854   
Latin             0.0837  0.0496       0.1074  0.1172   0.0403  0.0872   
Metal             0.1275  0.0924       0.0878  0.0769   0.0427  0.1150   
Pop               0.0893  0.0661       0.1062  0.0904   0.0499  0.0974   
Reggae/Caribbean  0.0820  0.0600       0.0924  0.0952   0.0464  0.0875   
Rock              0.1009  0.0718       0.1006  0.0870   0.0484  0.1053   
Soul/Funk         0.0804  0.0567      